In [ ]:
import polars as pl
import numpy as np
from scipy import stats
import altair as alt

In [ ]:
df = pl.scan_parquet("data/data_bias.parquet")

In [ ]:
def ci_lower(x, confidence = .95):
    x = x.to_numpy()
    n = len(x)

    mean = np.mean(x)
    se = stats.sem(x)
    ci = se * stats.t.ppf((1 + confidence) / 2, n-1)

    return mean - ci

def ci_upper(x, confidence = .95):
    x = x.to_numpy()
    n = len(x)

    mean = np.mean(x)
    se = stats.sem(x)
    ci = se * stats.t.ppf((1 + confidence) / 2, n-1)

    return mean + ci

In [ ]:
df.with_columns(((pl.col("adv distance") - pl.col("adv distance").mean()) / pl.col("adv distance").std()).over(["dataset", "bias_type"]))

In [ ]:
cols = ["train acc", "train f1", "test acc", "test f1", 
               "adv acc", "adv f1", "adv distance", "clusters"]

def agg_metrics(cols):
    res = []
    for col in cols:
        res.extend([
            pl.col(col).mean().alias(f"{col}_mean"),
            pl.col(col).std().alias(f"{col}_std"),
            pl.col(col).map_batches(ci_lower, return_dtype=pl.Float32, returns_scalar=True).alias(f"{col}_ci_lb"),
            pl.col(col).map_batches(ci_upper, return_dtype=pl.Float32, returns_scalar=True).alias(f"{col}_ci_ub"),
        ])
    return res

In [ ]:
df_stats = df.select(pl.exclude("seed")).with_columns(
    (
        (pl.col("adv distance") - pl.col("adv distance").mean()) / pl.col("adv distance").std()
        ).over(["dataset", "bias_type"])
    ).group_by(
    # pl.col("distribution"),
    # pl.col("model"),
    # pl.col("seed"),
    pl.col("bias"),
    pl.col("dataset"),
    pl.col("bias_type"),
    
).agg(
    agg_metrics(cols)
).sort([
    # pl.col("distribution"),
    # pl.col("seed"),
    # pl.col("depth"),
    pl.col("bias")
])

In [ ]:
df_stats.collect()

### Count vs density: which tracks bias?
Spearman trend of OPTICS cluster **count** and corrected **adv distance** (compactness/density) against the injected bias level, per `(dataset, bias_type)`. Computed on the per-fold rows. This is decision support only — the headline-metric choice is a research-design call.

In [ ]:
from scipy.stats import spearmanr

_trend = df.collect()
_rows = []
for key, g in _trend.group_by(["dataset", "bias_type"]):
    ds, bt = key
    b = g["bias"].to_numpy()
    rho_c, p_c = spearmanr(b, g["clusters"].to_numpy())
    rho_d, p_d = spearmanr(b, g["adv distance"].to_numpy())
    _rows.append({
        "dataset": ds, "bias_type": bt,
        "rho_count_vs_bias": round(rho_c, 3), "p_count": p_c,
        "rho_density_vs_bias": round(rho_d, 3), "p_density": p_d,
    })
spearman_summary = pl.DataFrame(_rows).sort(["dataset", "bias_type"])
spearman_summary

In [ ]:
base = alt.Chart(
    df_stats.collect()
)

scale = alt.Scale(
    domain=["train acc_mean", "test acc_mean", "clusters_mean"], 
    range=["blue", "red", "purple"]
)


dash_scale = alt.Scale(
    domain=["train acc_mean", "test acc_mean", "clusters_mean"],
    range=[[2, 4], [1, 0], [8, 4]]  # [dash_length, gap_length]
)

# Chart for accuracy metrics (left y-axis)
acc_chart = base.mark_line(strokeWidth=2).transform_fold(
    fold=["train acc_mean", "test acc_mean"],
    as_=["variable", "value"]
).encode(
    x=alt.X("bias:Q").title("Bias"),
    y=alt.Y('value:Q').title("Accuracy").scale(zero=False),
    color=alt.Color('variable:N', scale=scale, title="Metric"),
    strokeDash=alt.StrokeDash('variable:N', scale=dash_scale)
)


# Chart for adversarial distance (right y-axis)
adv_chart = base.mark_line(strokeWidth=2).transform_fold(
    fold=["clusters_mean"],
    as_=["variable", "value"]
).encode(
    x=alt.X("bias:Q").title("Bias"),
    y=alt.Y('value:Q').title("Adversarial Distance").axis(orient='right').scale(zero=False),
    color=alt.Color('variable:N', scale=scale, title="Metric"),
    strokeDash=alt.StrokeDash('variable:N', scale=dash_scale)

)

ci_band0 = base.mark_area(
    opacity=0.3,
    interpolate="basis",
).encode(
    x="bias:Q",
    y=alt.Y("train acc_ci_lb:Q").axis(title="Accuracy", orient="left"),
    y2=alt.Y2("train acc_ci_ub:Q"),
    color = alt.value("blue"),
)

ci_band1 = base.mark_area(
    opacity=0.3,
    interpolate="basis",
).encode(
    x="bias:Q",
    y=alt.Y("test acc_ci_lb:Q").axis(title="Accuracy", orient="left"),
    y2=alt.Y2("test acc_ci_ub:Q"),
    color = alt.value("red")
)

ci_band2 = base.mark_area(
    opacity=0.3,
    interpolate="basis",
).encode(
    x="bias:Q",
    y=alt.Y("adv distance_ci_lb:Q").axis(orient="right"),
    y2=alt.Y2("adv distance_ci_ub:Q"),
    color = alt.value("purple")
)

alt.layer(acc_chart + ci_band0 + ci_band1, adv_chart + ci_band2).resolve_scale(
    y='independent'
).facet(
    row="dataset",
    column="bias_type"
).resolve_scale(
    x="independent",
    y="independent"
)


### Adversarial density panel (additive, corrected)
The chart above folds `clusters_mean` (cluster **count**) under an axis titled *Adversarial Distance*, with a CI band drawn from the density columns — three different quantities on one line. The cell below plots the corrected `adv distance_mean` (per-cluster compactness/**density**) explicitly so count and density can be compared side by side. Higher density = tighter clusters. Note: the density definition `n_pairs / (total_separation + 1)` is size-confounded at the small cluster sizes here; mean pairwise separation is a scale-invariant alternative worth considering. Which becomes the headline metric is left to the humans.

In [ ]:
base2 = alt.Chart(df_stats.collect())

density_chart = base2.mark_line(strokeWidth=2, color="green").transform_fold(
    fold=["adv distance_mean"], as_=["variable", "value"]
).encode(
    x=alt.X("bias:Q").title("Bias"),
    y=alt.Y("value:Q").title("Adversarial density (compactness)").scale(zero=False),
)
density_band = base2.mark_area(opacity=0.25, color="green").encode(
    x="bias:Q",
    y=alt.Y("adv distance_ci_lb:Q").title("Adversarial density (compactness)"),
    y2="adv distance_ci_ub:Q",
)
count_chart = base2.mark_line(strokeWidth=2, color="purple").transform_fold(
    fold=["clusters_mean"], as_=["variable", "value"]
).encode(
    x=alt.X("bias:Q").title("Bias"),
    y=alt.Y("value:Q").title("Cluster count").axis(orient="right").scale(zero=False),
)
alt.layer(
    density_chart + density_band, count_chart
).resolve_scale(y="independent").facet(
    row="dataset", column="bias_type"
).resolve_scale(x="independent", y="independent")